## 스스로 해보자_코드 기록
### D006 - 데이터 전처리 자동화 - 반복 작업을 구조화 한다
---

In [ ]:
# 스스로 해보자! (1)
# 아래 주석(#)을 지우고 빈칸(___)을 채운 뒤 실행해보세요.

weight = {"basic": 1, "silver": 2, "gold": 3, "vip": 4}
orders["membership_score"] = orders["membership"].map(weight)   # 단순 대체니까 map이 괜찮음
print(orders[["membership", "membership_score"]].head())


orders["age_next_year_apply"] = orders["customer_age"].apply(lambda x: x+1 )
orders["age_next_year_vec"]   = orders["customer_age"] +  1
print((orders["age_next_year_apply"] == orders["age_next_year_vec"]).all()) # 이렇게 되면 기존 age 열의 결측치 제거가 안되서 null = null = false가 나와버림
print(orders["age_next_year_apply"].equals(orders["age_next_year_vec"]))  # 그래서 .equals()을 추가함 (.equals)은 NaN끼리는 같은 것으로 취급해서 비교

In [ ]:
# 스스로 해보자! (2)
# 1) channel_clean One-Hot
ch_oh = pd.get_dummies(orders["channel_clean"], prefix = "channel", dtype=int)
orders = pd.concat([orders, ch_oh], axis=1)
print(orders.filter(like="ch_").head())

# 2) drop_first 옵션
cat_oh = pd.get_dummies(orders["category"], prefix="cat", dtype=int, drop_first=___)
print(cat_oh.shape)   # 다중공선성으로 drop_frist

In [ ]:
# 스스로 해보자! (3)
# from sklearn.preprocessing import MinMaxScaler, RobustScaler

qty = orders[["quantity"]]
orders["quantity_mm"] = MinMaxScaler().fit_transform(qty)
orders["quantity_robust"] = RobustScaler().fit_transform(qty)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
sns.histplot(orders["quantity_mm"],     bins=30, ax=axes[0]); axes[0].set_title("Min-Max")
sns.histplot(orders["quantity_robust"], bins=30, ax=axes[1]); axes[1].set_title("Robust")
plt.tight_layout(); plt.show()

In [ ]:
# 스스로 해보자! (4)
# 아래 빈칸(___)을 채우고 실행해보세요.

result = (
    orders
    .dropna(subset=["amount"])
    .query("0 < customer_age < 120")
    .assign(
        channel_clean=lambda d: d["channel"].str.lower().str.strip(),
        amount_log=lambda d: np.log1p(d["amount"]),
    )
    .sort_values("amount", ascending=False)
    .reset_index(drop=True)
)
print(result.shape)
result.head(3)

In [ ]:
# 스스로 해보자! (5)
def add_amount_class(df):
    return df.assign(
        amount_class=np.where(df["amount"] >= 100_000, "high", "low")
    )

orders_v2 =(
orders
    .pipe(clean_strings)
    .pipe(drop_invalid)
    .pipe(add_features)
    .pipe(add_amount_class)
)  

print(orders_v2[["amount", "amount_class"]].head())
print(orders_v2["amount_class"].value_counts())